# Chapter 23: Model Context Protocol — The GAS Tool Layer

## 🎯 Chapter Goal
Build an MCP (Model Context Protocol) server exposing 5 GAS tools, connect a GAS Monitor Agent to it, and demonstrate the N+M benefit: adding a new tool requires only a server-side change.

## 📖 Theory Recap

### The N×M Problem
Without MCP, every agent must implement its own connector for every tool:
```
N agents × M tools = N×M integrations
5 agents × 10 tools = 50 custom connectors
```
With MCP, each agent connects to one MCP client, and each tool registers with one MCP server:
```
N agents + M tools = N+M integrations
5 agents + 10 tools = 15 registrations
```

### MCP Architecture
```
┌─────────────────────────────────────────────────────────┐
│                    MCP Client (Agent)                   │
│  - Discovers available tools via list_tools()           │
│  - Calls tools via call_tool(name, args)                │
└──────────────────────┬──────────────────────────────────┘
                       │  Transport (stdio / SSE / HTTP)
┌──────────────────────▼──────────────────────────────────┐
│                    MCP Server                           │
│  - Registers tools with @app.tool()                     │
│  - Handles tool invocation                              │
│  - Returns structured results                           │
└─────────────────────────────────────────────────────────┘
```

### MCP Primitives
| Primitive | Description |
|-----------|-------------|
| **Tools** | Functions the LLM can call (read/write side effects) |
| **Resources** | Data the LLM can read (files, DB records, APIs) |
| **Prompts** | Reusable prompt templates with parameters |
| **Sampling** | Server-side LLM calls (server requests completions) |

### Transport Options
- **stdio**: Server runs as subprocess, communicates via stdin/stdout. Best for local tools.
- **SSE (Server-Sent Events)**: HTTP-based streaming. Best for remote tools.
- **HTTP**: Standard request/response. Best for stateless tools.

### Tool Discovery Protocol
```json
// Client → Server
{"method": "tools/list"}

// Server → Client
{"tools": [{"name": "read_glucose_sensor", "description": "...", "inputSchema": {...}}]}

// Client → Server (tool call)
{"method": "tools/call", "params": {"name": "read_glucose_sensor", "arguments": {"patient_id": "P001"}}}
```

## 🔨 Build It

In [ ]:
import subprocess, sys
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "mcp", "langgraph", "langchain-openai"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ Dependencies installed")
else:
    print("⚠️  Some packages may not have installed. Continuing with mock implementations.")
    print(result.stderr[:200])

In [ ]:
import os
import json
import time
import random
import asyncio
from typing import Any, Optional
from dataclasses import dataclass

# ── Test-mode flag ──────────────────────────────────────────────────────────
import sys
sys.path.insert(0, os.path.join(os.getcwd(), "../.."))
from llm_utils import llm, TEST_MODE, BACKEND, MODEL

# ── Check MCP availability ───────────────────────────────────────────────────
try:
    import mcp
    MCP_AVAILABLE = True
    print(f"✅ MCP available: {mcp.__version__ if hasattr(mcp, '__version__') else 'installed'}")
except ImportError:
    MCP_AVAILABLE = False
    print("⚠️  MCP not available — using mock MCP implementation")

print(f"   TEST_MODE={TEST_MODE}, MCP_AVAILABLE={MCP_AVAILABLE}")

In [ ]:
# ── Mock MCP Framework (used when mcp package unavailable) ───────────────────
# This mirrors the real MCP API so the code structure is identical

class MockToolResult:
    def __init__(self, content):
        self.content = [{"type": "text", "text": json.dumps(content) if not isinstance(content, str) else content}]

class MockTool:
    def __init__(self, name: str, description: str, fn, input_schema: dict):
        self.name = name
        self.description = description
        self.fn = fn
        self.inputSchema = input_schema

class MockServer:
    """Mock MCP Server that mirrors the real mcp.Server API."""

    def __init__(self, name: str):
        self.name = name
        self._tools: dict[str, MockTool] = {}

    def tool(self):
        """Decorator to register a tool."""
        def decorator(fn):
            import inspect
            sig = inspect.signature(fn)
            schema = {"type": "object", "properties": {}, "required": []}
            for param_name, param in sig.parameters.items():
                ann = param.annotation
                type_map = {str: "string", int: "integer", float: "number", bool: "boolean", list: "array"}
                schema["properties"][param_name] = {"type": type_map.get(ann, "string")}
                if param.default is inspect.Parameter.empty:
                    schema["required"].append(param_name)
            tool = MockTool(
                name=fn.__name__,
                description=fn.__doc__ or "",
                fn=fn,
                input_schema=schema
            )
            self._tools[fn.__name__] = tool
            return fn
        return decorator

    async def list_tools(self) -> list:
        return list(self._tools.values())

    async def call_tool(self, name: str, arguments: dict) -> MockToolResult:
        if name not in self._tools:
            raise ValueError(f"Tool '{name}' not found")
        result = await self._tools[name].fn(**arguments)
        return MockToolResult(result)

class MockClient:
    """Mock MCP Client that connects to a MockServer."""

    def __init__(self, server: MockServer):
        self.server = server

    async def list_tools(self) -> list:
        return await self.server.list_tools()

    async def call_tool(self, name: str, arguments: dict) -> MockToolResult:
        return await self.server.call_tool(name, arguments)

print("✅ Mock MCP framework defined (mirrors real mcp.Server API)")

In [ ]:
# ── GAS MCP Server — 5 Tools ─────────────────────────────────────────────────

# Use real MCP if available, otherwise use mock
app = MockServer("gas-tools")
print("✅ Using MockServer (same API as mcp.Server)")

# ── Simulated patient database ───────────────────────────────────────────────
PATIENT_DB = {
    "P001": {"name": "Alice", "age": 45, "diabetes_type": "T1D", "caregiver": "bob@example.com"},
    "P002": {"name": "Bob",   "age": 62, "diabetes_type": "T2D", "caregiver": "carol@example.com"},
    "P003": {"name": "Carol", "age": 38, "diabetes_type": "T1D", "caregiver": "dave@example.com"},
}

GLUCOSE_HISTORY = {
    "P001": [95, 102, 115, 130, 145, 138, 120, 105, 98, 92],
    "P002": [140, 155, 170, 185, 195, 180, 165, 150, 140, 135],
    "P003": [85, 78, 70, 62, 55, 48, 65, 82, 95, 108],
}

ALERTS_SENT = []  # Track sent alerts

# ── Tool 1: read_glucose_sensor ──────────────────────────────────────────────
@app.tool()
async def read_glucose_sensor(patient_id: str) -> dict:
    """Read current CGM glucose value for a patient"""
    if patient_id not in PATIENT_DB:
        return {"error": f"Patient {patient_id} not found"}
    history = GLUCOSE_HISTORY.get(patient_id, [100])
    current = history[-1] + random.gauss(0, 2)
    trend_val = history[-1] - history[-2] if len(history) >= 2 else 0
    trend = "rising" if trend_val > 2 else ("falling" if trend_val < -2 else "stable")
    return {
        "patient_id": patient_id,
        "glucose_mg_dl": round(current, 1),
        "trend": trend,
        "trend_rate_mg_dl_per_min": round(trend_val / 5, 2),
        "timestamp": time.time(),
        "sensor_id": f"CGM-{patient_id}-001"
    }

# ── Tool 2: get_patient_history ──────────────────────────────────────────────
@app.tool()
async def get_patient_history(patient_id: str, hours: int) -> list:
    """Get glucose history for the last N hours"""
    if patient_id not in PATIENT_DB:
        return [{"error": f"Patient {patient_id} not found"}]
    history = GLUCOSE_HISTORY.get(patient_id, [])
    # Each reading is ~5 min apart; hours * 12 readings per hour
    n_readings = min(hours * 12, len(history))
    readings = history[-n_readings:]
    return [
        {
            "timestamp": time.time() - (len(readings) - i) * 300,
            "glucose_mg_dl": g,
            "index": i
        }
        for i, g in enumerate(readings)
    ]

# ── Tool 3: lookup_medical_guideline ────────────────────────────────────────
@app.tool()
async def lookup_medical_guideline(query: str) -> str:
    """Search ADA guidelines for a clinical question"""
    guidelines = {
        "hypoglycemia": "ADA 2024: Treat glucose < 70 mg/dL with 15g fast-acting carbs. Recheck in 15 min. If < 54 mg/dL, use glucagon.",
        "hyperglycemia": "ADA 2024: Glucose > 180 mg/dL post-meal requires assessment. Correction dose = (BG - target) / ISF.",
        "target range": "ADA 2024: Target 70-180 mg/dL for most adults with diabetes. Tighter targets (70-140) for pregnancy.",
        "correction dose": "Correction dose (units) = (Current BG - Target BG) / ISF. Typical ISF: 30-50 mg/dL per unit.",
        "exercise": "ADA 2024: Check BG before exercise. If < 90 mg/dL, consume 15-30g carbs. Reduce basal insulin 50% during exercise.",
        "dawn phenomenon": "ADA 2024: Dawn phenomenon (early morning hyperglycemia) managed with basal insulin adjustment or CSII.",
        "sick day": "ADA 2024: Check BG every 4h during illness. Never stop insulin. Check ketones if BG > 240 mg/dL.",
    }
    q = query.lower()
    for key, val in guidelines.items():
        if key in q:
            return val
    return f"No specific ADA guideline found for '{query}'. Consult ADA Standards of Medical Care in Diabetes 2024."

# ── Tool 4: calculate_correction_dose ───────────────────────────────────────
@app.tool()
async def calculate_correction_dose(current_glucose: float, target: float, isf: float) -> float:
    """Calculate insulin correction dose"""
    if isf <= 0:
        return 0.0
    dose = (current_glucose - target) / isf
    return round(max(0.0, dose), 2)  # No negative doses

# ── Tool 5: send_caregiver_alert ─────────────────────────────────────────────
@app.tool()
async def send_caregiver_alert(patient_id: str, message: str, severity: str) -> bool:
    """Send alert to registered caregiver"""
    if patient_id not in PATIENT_DB:
        return False
    caregiver = PATIENT_DB[patient_id]["caregiver"]
    alert = {
        "patient_id": patient_id,
        "caregiver": caregiver,
        "message": message,
        "severity": severity,
        "timestamp": time.time(),
        "sent": True
    }
    ALERTS_SENT.append(alert)
    print(f"  📧 Alert sent to {caregiver}: [{severity.upper()}] {message[:60]}")
    return True

print("✅ GAS MCP Server defined with 5 tools:")
print("   1. read_glucose_sensor")
print("   2. get_patient_history")
print("   3. lookup_medical_guideline")
print("   4. calculate_correction_dose")
print("   5. send_caregiver_alert")

In [ ]:
# ── MCP Client ───────────────────────────────────────────────────────────────

# Create client connected to our server
client = MockClient(app)

# ── Tool Discovery ───────────────────────────────────────────────────────────
async def demo_tool_discovery():
    print("🔍 Tool Discovery — client calls list_tools():")
    tools = await client.list_tools()
    print(f"   Found {len(tools)} tools:\n")
    for tool in tools:
        print(f"   📌 {tool.name}")
        print(f"      Description: {tool.description}")
        props = tool.inputSchema.get("properties", {})
        if props:
            print(f"      Parameters: {list(props.keys())}")
        print()
    return tools

tools = await demo_tool_discovery()

In [ ]:
# ── Connect GAS Monitor Agent to MCP Server ──────────────────────────────────

async def gas_monitor_agent_via_mcp(patient_id: str) -> dict:
    """
    GAS Monitor Agent that uses MCP tools to:
    1. Read current glucose
    2. Get history
    3. Look up guideline if needed
    4. Calculate correction dose if hyperglycemic
    5. Send caregiver alert if critical
    """
    print(f"\n🤖 GAS Monitor Agent running for patient {patient_id}")
    print("   (using MCP tools for all data access)")
    print("-" * 50)

    # Step 1: Read current glucose
    print("\n  Step 1: read_glucose_sensor")
    result = await client.call_tool("read_glucose_sensor", {"patient_id": patient_id})
    reading = json.loads(result.content[0]["text"])
    glucose = reading["glucose_mg_dl"]
    trend = reading["trend"]
    print(f"  → glucose={glucose} mg/dL, trend={trend}")

    # Step 2: Get history
    print("\n  Step 2: get_patient_history(hours=2)")
    result = await client.call_tool("get_patient_history", {"patient_id": patient_id, "hours": 2})
    history = json.loads(result.content[0]["text"])
    print(f"  → {len(history)} readings retrieved")

    # Classify alert level
    if glucose < 54 or glucose > 250:   alert = "critical"
    elif glucose < 70 or glucose > 180: alert = "high"
    elif glucose < 80 or glucose > 140: alert = "medium"
    else:                               alert = "none"

    recommendation = ""
    correction_dose = 0.0

    # Step 3: Look up guideline if alert
    if alert != "none":
        query = "hypoglycemia" if glucose < 70 else "hyperglycemia"
        print(f"\n  Step 3: lookup_medical_guideline('{query}')")
        result = await client.call_tool("lookup_medical_guideline", {"query": query})
        recommendation = result.content[0]["text"].strip('"')
        print(f"  → {recommendation[:80]}...")

    # Step 4: Calculate correction dose if hyperglycemic
    if glucose > 180:
        print("\n  Step 4: calculate_correction_dose")
        result = await client.call_tool(
            "calculate_correction_dose",
            {"current_glucose": glucose, "target": 120.0, "isf": 50.0}
        )
        correction_dose = float(result.content[0]["text"])
        print(f"  → correction dose = {correction_dose}U")

    # Step 5: Send caregiver alert if critical
    notified = False
    if alert == "critical":
        print("\n  Step 5: send_caregiver_alert")
        msg = f"CRITICAL: glucose={glucose} mg/dL, trend={trend}. {recommendation[:60]}"
        result = await client.call_tool(
            "send_caregiver_alert",
            {"patient_id": patient_id, "message": msg, "severity": "critical"}
        )
        notified = json.loads(result.content[0]["text"])

    return {
        "patient_id": patient_id,
        "glucose": glucose,
        "trend": trend,
        "alert_level": alert,
        "recommendation": recommendation,
        "correction_dose": correction_dose,
        "caregiver_notified": notified,
    }

# Run for 3 patients
async def run_all_patients():
    results = []
    for patient_id in ["P001", "P002", "P003"]:
        result = await gas_monitor_agent_via_mcp(patient_id)
        results.append(result)
    return results

results = await run_all_patients()

print("\n" + "=" * 50)
print("📊 Summary:")
print(f"{'Patient':<10} {'Glucose':>8} {'Alert':>10} {'Notified':>10}")
print("-" * 42)
for r in results:
    print(f"{r['patient_id']:<10} {r['glucose']:>8.1f} {r['alert_level']:>10} {str(r['caregiver_notified']):>10}")

In [ ]:
# ── The N+M Benefit: Adding a New Tool ──────────────────────────────────────

print("=" * 60)
print("THE N+M BENEFIT: Adding a New Tool")
print("=" * 60)
print()
print("Without MCP: adding check_drug_interactions requires updating")
print("ALL 5 GAS agents (5 code changes).")
print()
print("With MCP: add ONE tool to the server. All agents get it automatically.")
print()

# Add a 6th tool to the server — no agent code changes needed
@app.tool()
async def check_drug_interactions(medication: str, glucose: float) -> dict:
    """Check drug-glucose interactions for a given medication"""
    interactions = {
        "metformin": {"effect": "neutral", "hypoglycemia_risk": False, "note": "Safe to use alone"},
        "insulin": {"effect": "lowers", "hypoglycemia_risk": True, "note": "Monitor for hypoglycemia"},
        "glipizide": {"effect": "lowers", "hypoglycemia_risk": True, "note": "Sulfonylurea — skip if meal skipped"},
        "prednisone": {"effect": "raises", "hypoglycemia_risk": False, "note": "Steroid-induced hyperglycemia"},
        "lisinopril": {"effect": "neutral", "hypoglycemia_risk": False, "note": "ACE inhibitor — no glucose effect"},
    }
    for med, info in interactions.items():
        if med in medication.lower():
            return {**info, "medication": medication, "current_glucose": glucose}
    return {"effect": "unknown", "hypoglycemia_risk": False, "note": "No interaction data", "medication": medication}

print("✅ Added tool 6: check_drug_interactions (server-side only change)")
print()

# Verify all agents can now discover the new tool
async def verify_new_tool():
    tools = await client.list_tools()
    print(f"   Client now sees {len(tools)} tools (was 5, now 6):")
    for t in tools:
        print(f"   - {t.name}")

    # Call the new tool
    print("\n   Testing new tool:")
    result = await client.call_tool(
        "check_drug_interactions",
        {"medication": "insulin", "glucose": 58.0}
    )
    data = json.loads(result.content[0]["text"])
    print(f"   → {data}")

await verify_new_tool()

print()
print("💡 N+M math:")
print("   Without MCP: 5 agents × 6 tools = 30 integrations")
print("   With MCP:    5 agents + 6 tools = 11 registrations")
print("   Savings: 19 fewer integrations (63% reduction)")

In [ ]:
# ── MCP Server as Standalone Script ─────────────────────────────────────────
# This shows what a real MCP server script looks like

MCP_SERVER_SCRIPT = '''
#!/usr/bin/env python3
"""GAS MCP Server — run as: python gas_mcp_server.py"""

import asyncio
from mcp.server import Server
from mcp.server.stdio import stdio_server

app = Server("gas-tools")

@app.tool()
async def read_glucose_sensor(patient_id: str) -> dict:
    """Read current CGM glucose value for a patient"""
    # In production: connect to real CGM API
    return {"patient_id": patient_id, "glucose_mg_dl": 105.0, "trend": "stable"}

@app.tool()
async def send_caregiver_alert(patient_id: str, message: str, severity: str) -> bool:
    """Send alert to registered caregiver"""
    # In production: send SMS/email via Twilio/SendGrid
    print(f"ALERT [{severity}] for {patient_id}: {message}")
    return True

async def main():
    async with stdio_server() as (read_stream, write_stream):
        await app.run(read_stream, write_stream, app.create_initialization_options())

if __name__ == "__main__":
    await main()
'''

print("📄 Real MCP Server script (gas_mcp_server.py):")
print(MCP_SERVER_SCRIPT)

# Save to file for reference
with open("/tmp/gas_mcp_server.py", "w") as f:
    f.write(MCP_SERVER_SCRIPT)
print("✅ Saved to /tmp/gas_mcp_server.py")
print("   Run with: python /tmp/gas_mcp_server.py")
print("   Connect with: mcp connect stdio python /tmp/gas_mcp_server.py")

## 🏥 GAS Connection

All 5 GAS agents access sensors, EHR data, and clinical guidelines through the **MCP server**. This standardizes tool access and makes the system extensible:

| GAS Agent | MCP Tools Used |
|-----------|---------------|
| Monitor Agent | `read_glucose_sensor`, `get_patient_history` |
| Advisor Agent | `lookup_medical_guideline`, `calculate_correction_dose` |
| Safety Critic | `lookup_medical_guideline`, `check_drug_interactions` |
| Caregiver Notifier | `send_caregiver_alert` |
| Supervisor Agent | `read_glucose_sensor` (for routing decisions) |

**Architecture benefits:**
1. **Standardization**: All agents use the same tool interface — no custom connectors
2. **Extensibility**: Adding a new data source (e.g., EHR integration) requires only a new MCP tool
3. **Security**: MCP server can enforce access control — agents only see tools they're authorized for
4. **Observability**: All tool calls flow through one server — easy to log, audit, and monitor
5. **Testability**: Mock the MCP server in tests — agents don't need real sensors

## ✅ Checkpoint

**What we built:**
- ✅ **GAS MCP Server** with 5 tools: `read_glucose_sensor`, `get_patient_history`, `lookup_medical_guideline`, `calculate_correction_dose`, `send_caregiver_alert`
- ✅ **Mock MCP framework** that mirrors the real `mcp.Server` API — works without the package installed
- ✅ **MCP Client** that discovers tools via `list_tools()` and calls them via `call_tool()`
- ✅ **GAS Monitor Agent** connected to MCP — uses 5 tool calls to monitor 3 patients
- ✅ **N+M benefit demonstrated**: added `check_drug_interactions` as tool #6 — zero agent code changes
- ✅ **Standalone server script** showing real `stdio_server` usage

**Key insight:** MCP transforms the GAS tool layer from N×M custom integrations to N+M standardized registrations. As GAS grows from 5 to 50 tools, this becomes increasingly valuable.

**Next:** Chapter 24 — Agent Skills (building the GAS skill library)